In [4]:
import pandas as pd
from sqlalchemy import create_engine

# Cambia estos valores por los tuyos
USER = "postgres.irjzhqrawpjqcicmbwwc"
PASSWORD = "los4fantasticos*"
HOST = "aws-1-us-east-1.pooler.supabase.com"
PORT = "5432"
DBNAME = "postgres"

engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")

query = """
SELECT *
FROM f1_simulation_laptime_ml;
"""

df = pd.read_sql(query, engine)

print(df.shape)
print(df.head())
print(df.dtypes)

(142342, 26)
   Year                        Event Driver             Team  LapNumber  \
0  2020  70th Anniversary Grand Prix    BOT         Mercedes         39   
1  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         52   
2  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         50   
3  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         49   
4  2020  70th Anniversary Grand Prix    ALB  Red Bull Racing         48   

   Stint Compound  TyreLife  FreshTyre TrackStatus  ...  SpeedFL  SpeedST  \
0      3     HARD         7       True           1  ...    250.0    306.0   
1      3     HARD        22       True           1  ...    247.0    261.0   
2      3     HARD        20       True           1  ...    251.0    301.0   
3      3     HARD        19       True           1  ...    254.0    324.0   
4      3     HARD        18       True           1  ...    253.0    300.0   

   lap_position    AirTemp   Humidity     Pressure  Rainfall  TrackTemp  

In [5]:
from sklearn.model_selection import train_test_split

# Variable objetivo
y = df["laptime_seconds"]

# Variables de entrada
X = df.drop(columns=[
    "laptime_seconds",
    "sector1_seconds",
    "sector2_seconds",
    "sector3_seconds"
], errors="ignore")

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())


X shape: (142342, 22)
y shape: (142342,)


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,...,SpeedFL,SpeedST,lap_position,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7,True,1,...,250.0,306.0,3,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,52,3,HARD,22,True,1,...,247.0,261.0,5,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,50,3,HARD,20,True,1,...,251.0,301.0,5,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,49,3,HARD,19,True,1,...,254.0,324.0,6,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,48,3,HARD,18,True,1,...,253.0,300.0,6,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [6]:
query = """
SELECT *
FROM f1_simulation_laptime_ml;
"""

df = pd.read_sql(query, engine)

print(df.shape)
print(df["Year"].value_counts().sort_index())
display(df.head())

(142342, 26)
Year
2020    19926
2021    24347
2022    21860
2023    49828
2024    26381
Name: count, dtype: int64


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,...,SpeedFL,SpeedST,lap_position,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7,True,1,...,250.0,306.0,3,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,52,3,HARD,22,True,1,...,247.0,261.0,5,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,50,3,HARD,20,True,1,...,251.0,301.0,5,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,49,3,HARD,19,True,1,...,254.0,324.0,6,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,48,3,HARD,18,True,1,...,253.0,300.0,6,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [7]:
from sklearn.model_selection import train_test_split

y = df["laptime_seconds"]

X = df.drop(columns=[
    "laptime_seconds",
    "sector1_seconds",
    "sector2_seconds",
    "sector3_seconds"
], errors="ignore")

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

X shape: (142342, 22)
y shape: (142342,)
Numéricas: ['Year', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'lap_position', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']
Categóricas: ['Event', 'Driver', 'Team', 'Compound', 'TrackStatus']


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.660052435982999
RMSE: 2.0346091146944323
R2: 0.9958989712954118


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RepeatedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# 1. Definir X e y
# =========================================================

y = df["laptime_seconds"]

X_sim = df.drop(columns=[
    "laptime_seconds",
    "sector1_seconds",
    "sector2_seconds",
    "sector3_seconds",
    "SpeedI1",
    "SpeedI2",
    "SpeedFL",
    "SpeedST",
    "lap_position"
], errors="ignore")

print("X_sim shape:", X_sim.shape)
print("y shape:", y.shape)

# =========================================================
# 2. Separar primero el 20% para TEST
# =========================================================

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_sim, y,
    test_size=0.20,
    random_state=42
)

# =========================================================
# 3. Del 80% restante:
#    70% TRAIN
#    30% VAL
# =========================================================

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.30,
    random_state=42
)

print("\nTamaños:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

# =========================================================
# 4. Identificar columnas usando TRAIN
# =========================================================

numeric_features_sim = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_sim = X_train.select_dtypes(include=["object"]).columns.tolist()

print("\nNuméricas simulación:", numeric_features_sim)
print("Categóricas simulación:", categorical_features_sim)

# =========================================================
# 5. Preprocesamiento
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_sim = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_sim),
        ("cat", categorical_transformer, categorical_features_sim)
    ]
)

# =========================================================
# 6. Modelo
# =========================================================

model_sim = Pipeline(steps=[
    ("preprocessor", preprocessor_sim),
    ("regressor", RandomForestRegressor(
        n_estimators=30,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

# =========================================================
# 7. Cross validation SOLO en TRAIN
# =========================================================

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

cv_results = cross_validate(
    estimator=model_sim,
    X=X_train,
    y=y_train,
    cv=rkf,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    n_jobs=-1,
    return_train_score=False
)

cv_mae = -cv_results["test_mae"]
cv_rmse = -cv_results["test_rmse"]
cv_r2 = cv_results["test_r2"]

print("\nCross Validation SOLO en TRAIN")
print("MAE CV promedio :", cv_mae.mean())
print("MAE CV std      :", cv_mae.std())
print("RMSE CV promedio:", cv_rmse.mean())
print("RMSE CV std     :", cv_rmse.std())
print("R2 CV promedio  :", cv_r2.mean())
print("R2 CV std       :", cv_r2.std())

# =========================================================
# 8. Entrenar SOLO con TRAIN
# =========================================================

model_sim.fit(X_train, y_train)

# =========================================================
# 9. Evaluar en VALIDACIÓN
# =========================================================

y_val_pred = model_sim.predict(X_val)

mae_val = mean_absolute_error(y_val, y_val_pred)
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2_val = r2_score(y_val, y_val_pred)

print("\nMétricas en VALIDACIÓN")
print("MAE val :", mae_val)
print("RMSE val:", rmse_val)
print("R2 val  :", r2_val)

# =========================================================
# 10. Evaluar al final en TEST
# =========================================================

y_test_pred = model_sim.predict(X_test)

mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print("\nMétricas en TEST")
print("MAE test :", mae_test)
print("RMSE test:", rmse_test)
print("R2 test  :", r2_test)

X_sim shape: (142342, 17)
y shape: (142342,)

Tamaños:
Train: (79711, 17) (79711,)
Val  : (34162, 17) (34162,)
Test : (28469, 17) (28469,)

Numéricas simulación: ['Year', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']
Categóricas simulación: ['Event', 'Driver', 'Team', 'Compound', 'TrackStatus']

Cross Validation SOLO en TRAIN
MAE CV promedio : 1.8575983499442432
MAE CV std      : 0.06659356543366131
RMSE CV promedio: 3.951146464664721
RMSE CV std     : 0.17480265404255754
R2 CV promedio  : 0.9708336588748147
R2 CV std       : 0.019162389353853067

Métricas en VALIDACIÓN
MAE val : 1.8358362049853363
RMSE val: 3.7290132053532488
R2 val  : 0.9868611255948634

Métricas en TEST
MAE test : 1.8345436367758818
RMSE test: 3.746367447050687
R2 test  : 0.9860956215517404


In [16]:
df_base = X_sim

feature_cols = X_sim.columns.tolist()

base_rows = df_base[
    (df_base["Event"] == "70th Anniversary Grand Prix") &
    (df_base["LapNumber"] % 10 == 0) &
    (df_base["Rainfall"] == False) &
    (df_base["Compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()

base_rows = base_rows.sort_values(["Driver", "LapNumber"])

display(base_rows[[
    "Driver", "Event", "LapNumber", "Stint",
    "Compound", "TyreLife", "TrackTemp", "Rainfall"
]])

,Driver,Event,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall
51,ALB,70th Anniversary Grand Prix,10,2,HARD,4,43.227586,False
21,ALB,70th Anniversary Grand Prix,20,2,HARD,14,43.227586,False
36,ALB,70th Anniversary Grand Prix,30,2,HARD,24,43.227586,False
12,ALB,70th Anniversary Grand Prix,40,3,HARD,10,43.227586,False
2,ALB,70th Anniversary Grand Prix,50,3,HARD,20,43.227586,False
...,...,...,...,...,...,...,...,...
642,VET,70th Anniversary Grand Prix,10,1,HARD,11,43.227586,False
670,VET,70th Anniversary Grand Prix,20,1,HARD,21,43.227586,False
2278,VET,70th Anniversary Grand Prix,30,2,HARD,8,43.227586,False
2298,VET,70th Anniversary Grand Prix,40,3,MEDIUM,10,43.227586,False


In [17]:
compound_options = ["SOFT", "MEDIUM", "HARD"]
scenarios_compound = []

for _, base_row in base_rows.iterrows():
    for comp in compound_options:
        row = base_row[feature_cols].copy()
        row["Compound"] = comp
        row["Rainfall"] = False

        row_df = pd.DataFrame([row], columns=feature_cols)
        pred = model_sim.predict(row_df)[0]

        out = base_row.copy()
        out["scenario_type"] = "compound_dry"
        out["scenario_value"] = comp
        out["simulated_compound"] = comp
        out["predicted_laptime"] = pred

        scenarios_compound.append(out)

sim_compound_dry = pd.DataFrame(scenarios_compound)

display(
    sim_compound_dry[[
        "Driver", "Event", "LapNumber", "Stint",
        "Compound", "simulated_compound",
        "TyreLife", "TrackTemp", "Rainfall",
        "predicted_laptime"
    ]].sort_values(["Driver", "LapNumber", "simulated_compound"])
)

,Driver,Event,LapNumber,Stint,Compound,simulated_compound,TyreLife,TrackTemp,Rainfall,predicted_laptime
51,ALB,70th Anniversary Grand Prix,10,2,HARD,HARD,4,43.227586,False,93.192986
51,ALB,70th Anniversary Grand Prix,10,2,HARD,MEDIUM,4,43.227586,False,93.192986
51,ALB,70th Anniversary Grand Prix,10,2,HARD,SOFT,4,43.227586,False,93.714347
21,ALB,70th Anniversary Grand Prix,20,2,HARD,HARD,14,43.227586,False,93.149281
21,ALB,70th Anniversary Grand Prix,20,2,HARD,MEDIUM,14,43.227586,False,93.130543
...,...,...,...,...,...,...,...,...,...,...
2298,VET,70th Anniversary Grand Prix,40,3,MEDIUM,MEDIUM,10,43.227586,False,91.470592
2298,VET,70th Anniversary Grand Prix,40,3,MEDIUM,SOFT,10,43.227586,False,95.477016
2288,VET,70th Anniversary Grand Prix,50,3,MEDIUM,HARD,20,43.227586,False,91.372322
2288,VET,70th Anniversary Grand Prix,50,3,MEDIUM,MEDIUM,20,43.227586,False,91.372322


In [18]:
# =========================
# Escenario 2: degradación del neumático
# =========================

tyre_life_options = [1, 5, 10, 15, 20, 25]
scenarios_tyre = []

# usar exactamente las columnas con las que se entrenó el modelo
feature_cols = X_sim.columns.tolist()

for _, base_row in base_rows.iterrows():
    for tl in tyre_life_options:
        row = base_row[feature_cols].copy()
        row["Compound"] = "HARD"   # dejamos fijo el compuesto base
        row["Rainfall"] = False
        row["TyreLife"] = tl

        row_df = pd.DataFrame([row], columns=feature_cols)
        pred = model_sim.predict(row_df)[0]

        out = base_row.copy()
        out["Compound"] = "HARD"
        out["TyreLife"] = tl
        out["scenario_type"] = "tyre_life"
        out["scenario_value"] = tl
        out["predicted_laptime"] = pred

        scenarios_tyre.append(out)

sim_tyre = pd.DataFrame(scenarios_tyre)

display(
    sim_tyre[[
        "Driver", "Event", "LapNumber", "Stint",
        "Compound", "TyreLife", "TrackTemp", "Rainfall",
        "predicted_laptime", "scenario_type", "scenario_value"
    ]].sort_values(["Driver", "LapNumber", "TyreLife"])
)

,Driver,Event,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall,predicted_laptime,scenario_type,scenario_value
51,ALB,70th Anniversary Grand Prix,10,2,HARD,1,43.227586,False,112.254171,tyre_life,1
51,ALB,70th Anniversary Grand Prix,10,2,HARD,5,43.227586,False,93.208159,tyre_life,5
51,ALB,70th Anniversary Grand Prix,10,2,HARD,10,43.227586,False,93.246973,tyre_life,10
51,ALB,70th Anniversary Grand Prix,10,2,HARD,15,43.227586,False,93.086833,tyre_life,15
51,ALB,70th Anniversary Grand Prix,10,2,HARD,20,43.227586,False,92.822205,tyre_life,20
...,...,...,...,...,...,...,...,...,...,...,...
2288,VET,70th Anniversary Grand Prix,50,3,HARD,5,43.227586,False,91.391575,tyre_life,5
2288,VET,70th Anniversary Grand Prix,50,3,HARD,10,43.227586,False,91.391575,tyre_life,10
2288,VET,70th Anniversary Grand Prix,50,3,HARD,15,43.227586,False,91.391575,tyre_life,15
2288,VET,70th Anniversary Grand Prix,50,3,HARD,20,43.227586,False,91.372322,tyre_life,20


In [20]:
# =========================
# Escenario 3: lluvia
# =========================

rain_options = [False, True]
scenarios_rain = []

feature_cols = X_sim.columns.tolist()

for _, base_row in base_rows.iterrows():
    for rain in rain_options:
        row = base_row[feature_cols].copy()
        row["Rainfall"] = rain

        row_df = pd.DataFrame([row], columns=feature_cols)
        pred = model_sim.predict(row_df)[0]

        out = base_row.copy()
        out["Rainfall"] = rain
        out["scenario_type"] = "rainfall"
        out["scenario_value"] = rain
        out["predicted_laptime"] = pred

        scenarios_rain.append(out)

sim_rain = pd.DataFrame(scenarios_rain)

display(
    sim_rain[[
        "Driver", "Event", "LapNumber", "Stint",
        "Compound", "TyreLife", "TrackTemp", "Rainfall",
        "predicted_laptime", "scenario_type", "scenario_value"
    ]].sort_values(["Driver", "LapNumber", "Rainfall"])
)

,Driver,Event,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall,predicted_laptime,scenario_type,scenario_value
51,ALB,70th Anniversary Grand Prix,10,2,HARD,4,43.227586,False,93.192986,rainfall,False
51,ALB,70th Anniversary Grand Prix,10,2,HARD,4,43.227586,True,94.764827,rainfall,True
21,ALB,70th Anniversary Grand Prix,20,2,HARD,14,43.227586,False,93.149281,rainfall,False
21,ALB,70th Anniversary Grand Prix,20,2,HARD,14,43.227586,True,94.803041,rainfall,True
36,ALB,70th Anniversary Grand Prix,30,2,HARD,24,43.227586,False,91.323227,rainfall,False
...,...,...,...,...,...,...,...,...,...,...,...
2278,VET,70th Anniversary Grand Prix,30,2,HARD,8,43.227586,True,93.362479,rainfall,True
2298,VET,70th Anniversary Grand Prix,40,3,MEDIUM,10,43.227586,False,91.470592,rainfall,False
2298,VET,70th Anniversary Grand Prix,40,3,MEDIUM,10,43.227586,True,93.041483,rainfall,True
2288,VET,70th Anniversary Grand Prix,50,3,MEDIUM,20,43.227586,False,91.372322,rainfall,False


In [21]:
realistic_scenarios = [
    {"Compound": "SOFT", "TyreLife": 2,  "FreshTyre": True,  "Rainfall": False, "TrackTemp": 35.0},
    {"Compound": "MEDIUM", "TyreLife": 8,  "FreshTyre": False, "Rainfall": False, "TrackTemp": 35.0},
    {"Compound": "HARD", "TyreLife": 15, "FreshTyre": False, "Rainfall": False, "TrackTemp": 35.0},
    {"Compound": "INTERMEDIATE", "TyreLife": 5, "FreshTyre": False, "Rainfall": True, "TrackTemp": 25.0},
    {"Compound": "WET", "TyreLife": 5, "FreshTyre": False, "Rainfall": True, "TrackTemp": 20.0}
]

scenarios_realistic = []

feature_cols = X_sim.columns.tolist()

for _, base_row in base_rows.iterrows():
    for s in realistic_scenarios:
        row = base_row[feature_cols].copy()

        for k, v in s.items():
            row[k] = v

        row_df = pd.DataFrame([row], columns=feature_cols)
        pred = model_sim.predict(row_df)[0]

        out = base_row.copy()
        for k, v in s.items():
            out[k] = v

        out["scenario_type"] = "realistic_combined"
        out["scenario_value"] = str(s)
        out["predicted_laptime"] = pred

        scenarios_realistic.append(out)

sim_realistic = pd.DataFrame(scenarios_realistic)

display(
    sim_realistic[[
        "Driver", "Event", "LapNumber", "Stint",
        "Compound", "TyreLife", "FreshTyre",
        "TrackTemp", "Rainfall",
        "predicted_laptime", "scenario_value"
    ]].sort_values(["Driver", "LapNumber", "Compound"])
)

,Driver,Event,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackTemp,Rainfall,predicted_laptime,scenario_value
51,ALB,70th Anniversary Grand Prix,10,2,HARD,15,False,35.0,False,92.564251,"{'Compound': 'HARD', 'TyreLife': 15, 'FreshTyr..."
51,ALB,70th Anniversary Grand Prix,10,2,INTERMEDIATE,5,False,25.0,True,99.458250,"{'Compound': 'INTERMEDIATE', 'TyreLife': 5, 'F..."
51,ALB,70th Anniversary Grand Prix,10,2,MEDIUM,8,False,35.0,False,92.705653,"{'Compound': 'MEDIUM', 'TyreLife': 8, 'FreshTy..."
51,ALB,70th Anniversary Grand Prix,10,2,SOFT,2,True,35.0,False,93.555186,"{'Compound': 'SOFT', 'TyreLife': 2, 'FreshTyre..."
51,ALB,70th Anniversary Grand Prix,10,2,WET,5,False,20.0,True,99.292524,"{'Compound': 'WET', 'TyreLife': 5, 'FreshTyre'..."
...,...,...,...,...,...,...,...,...,...,...,...
2288,VET,70th Anniversary Grand Prix,50,3,HARD,15,False,35.0,False,90.781732,"{'Compound': 'HARD', 'TyreLife': 15, 'FreshTyr..."
2288,VET,70th Anniversary Grand Prix,50,3,INTERMEDIATE,5,False,25.0,True,94.914123,"{'Compound': 'INTERMEDIATE', 'TyreLife': 5, 'F..."
2288,VET,70th Anniversary Grand Prix,50,3,MEDIUM,8,False,35.0,False,90.781732,"{'Compound': 'MEDIUM', 'TyreLife': 8, 'FreshTy..."
2288,VET,70th Anniversary Grand Prix,50,3,SOFT,2,True,35.0,False,94.727773,"{'Compound': 'SOFT', 'TyreLife': 2, 'FreshTyre..."


In [22]:
import uuid

scenario_run_id = f"scenario_{uuid.uuid4().hex[:12]}"
print("scenario_run_id:", scenario_run_id)

sim_realistic_to_save = sim_realistic.copy()
sim_realistic_to_save["scenario_run_id"] = scenario_run_id

# Si scenario_value no es texto, convertirlo
sim_realistic_to_save["scenario_value"] = sim_realistic_to_save["scenario_value"].astype(str)

# Columnas esperadas para guardar
cols = [
    "scenario_run_id", "Year", "Event", "Driver", "Team", "LapNumber", "Stint",
    "Compound", "TyreLife", "FreshTyre", "TrackStatus",
    "AirTemp", "Humidity", "Pressure", "Rainfall",
    "TrackTemp", "WindDirection", "WindSpeed",
    "scenario_type", "scenario_value", "predicted_laptime"
]

# Crear columnas faltantes si no existen
for c in cols:
    if c not in sim_realistic_to_save.columns:
        sim_realistic_to_save[c] = None

# Reordenar
sim_realistic_to_save = sim_realistic_to_save[cols]

# Guardar en Supabase
sim_realistic_to_save.to_sql(
    "simulation_scenarios",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Escenarios guardados en Supabase.")
print("Filas guardadas:", len(sim_realistic_to_save))

scenario_run_id: scenario_aa81709e2dd8
Escenarios guardados en Supabase.
Filas guardadas: 495


In [ ]:
query_final = """
SELECT *
FROM f1_simulation_base;
"""

df_final = pd.read_sql(query_final, engine)

print(df_final.shape)
display(df_final.head())
print(df_final.dtypes)

(145676, 34)


,Year,Event,Driver,DriverNumber,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,...,final_position,Points,Status,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,77,Mercedes,39,3,HARD,7.0,True,...,3.0,15.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
1,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,52,3,HARD,22.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
2,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,50,3,HARD,20.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
3,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,49,3,HARD,19.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207
4,2020,70th Anniversary Grand Prix,ALB,23,Red Bull Racing,48,3,HARD,18.0,True,...,5.0,10.0,Finished,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


Year                        int64
Event                      object
Driver                     object
DriverNumber                int64
Team                       object
LapNumber                   int64
Stint                       int64
Compound                   object
TyreLife                  float64
FreshTyre                    bool
TrackStatus                object
LapTime           timedelta64[ns]
Sector1Time       timedelta64[ns]
Sector2Time       timedelta64[ns]
Sector3Time       timedelta64[ns]
SpeedI1                   float64
SpeedI2                   float64
SpeedFL                   float64
SpeedST                   float64
lap_position              float64
BroadcastName              object
Abbreviation               object
TeamName                   object
GridPosition              float64
final_position            float64
Points                    float64
Status                     object
AirTemp                   float64
Humidity                  float64
Pressure      

In [ ]:
check_metrics = pd.read_sql("""
SELECT *
FROM model_metrics
ORDER BY created_at DESC
LIMIT 10;
""", engine)

display(check_metrics)

,id,run_id,model_name,target,created_at,train_rows,val_rows,test_rows,mccv_mae_mean,mccv_rmse_mean,mccv_r2_mean,val_mae,val_rmse,val_r2,test_mae,test_rmse,test_r2,notes
0,2,finalpos_f35db52881,RandomForestRegressor_final_position_v1,final_position,2026-03-07 23:02:28.768357,66679,28577,23814,0.679921,1.20501,0.94991,NaN,NaN,NaN,0.639759,1.136136,0.955471,Predicción de posición final a nivel de vuelta...
1,1,laptime_7653b18060,RandomForestRegressor_realistic_v1,laptime_seconds,2026-03-07 23:00:50.401292,79711,34162,28469,0.679921,1.20501,0.94991,1.835836,3.729013,0.986861,1.834544,3.746367,0.986096,"Modelo realista sin sectores, sin velocidades ..."


In [ ]:
base_final = X_final.iloc[[0]].copy()

print("Fila base para final_position:")
display(base_final)

Fila base para final_position:


,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,GridPosition,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,1.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [ ]:
base_final = X_final[
    (X_final["Compound"].isin(["SOFT", "MEDIUM", "HARD"])) &
    (X_final["Rainfall"] == False)
].iloc[[0]].copy()

display(base_final)

,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,TrackStatus,GridPosition,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,2020,70th Anniversary Grand Prix,BOT,Mercedes,39,3,HARD,7.0,True,1,1.0,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207


In [ ]:
grid_options = [1, 3, 5, 10, 15, 20]

scenarios_grid = []

for gp in grid_options:
    row = base_final.copy()
    row["GridPosition"] = gp

    pred = model_final.predict(row)[0]

    out = row.copy()
    out["scenario_type"] = "grid_position"
    out["scenario_value"] = gp
    out["predicted_final_position"] = pred
    scenarios_grid.append(out)

sim_grid = pd.concat(scenarios_grid, ignore_index=True)

display(sim_grid[[
    "Driver", "Event", "LapNumber", "Compound",
    "TyreLife", "GridPosition", "Rainfall",
    "predicted_final_position"
]])

,Driver,Event,LapNumber,Compound,TyreLife,GridPosition,Rainfall,predicted_final_position
0,BOT,70th Anniversary Grand Prix,39,HARD,7.0,1,False,2.134202
1,BOT,70th Anniversary Grand Prix,39,HARD,7.0,3,False,4.335040
2,BOT,70th Anniversary Grand Prix,39,HARD,7.0,5,False,4.528746
3,BOT,70th Anniversary Grand Prix,39,HARD,7.0,10,False,4.133333
4,BOT,70th Anniversary Grand Prix,39,HARD,7.0,15,False,5.400000
5,BOT,70th Anniversary Grand Prix,39,HARD,7.0,20,False,5.400000


In [23]:
print(sorted(df["Team"].dropna().unique()))
print(sorted(df["Driver"].dropna().unique()))

['Alfa Romeo', 'Alfa Romeo Racing', 'AlphaTauri', 'Alpine', 'Aston Martin', 'Ferrari', 'Haas F1 Team', 'Kick Sauber', 'McLaren', 'Mercedes', 'RB', 'Racing Point', 'Red Bull Racing', 'Renault', 'Williams']
['AIT', 'ALB', 'ALO', 'BEA', 'BOT', 'COL', 'DEV', 'DOO', 'FIT', 'GAS', 'GIO', 'GRO', 'HAM', 'HUL', 'KUB', 'KVY', 'LAT', 'LAW', 'LEC', 'MAG', 'MAZ', 'MSC', 'NOR', 'OCO', 'PER', 'PIA', 'RAI', 'RIC', 'RUS', 'SAI', 'SAR', 'STR', 'TSU', 'VER', 'VET', 'ZHO']


In [25]:
driver_team_pairs = (
    X_sim[["Driver", "Team"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["Team", "Driver"])
)

display(driver_team_pairs)

,Driver,Team
30711,BOT,Alfa Romeo
32307,ZHO,Alfa Romeo
94,GIO,Alfa Romeo Racing
17876,KUB,Alfa Romeo Racing
455,RAI,Alfa Romeo Racing
47446,DEV,AlphaTauri
66,GAS,AlphaTauri
194,KVY,AlphaTauri
56985,LAW,AlphaTauri
46411,RIC,AlphaTauri


In [28]:
# Ver los eventos únicos por año
events_per_year = (
    X_sim[["Year", "Event"]]
    .drop_duplicates()
    .sort_values(["Year", "Event"])
)

# Tomar 5 eventos por cada año
selected_events_per_year = (
    events_per_year
    .groupby("Year", as_index=False)
    .head(5)
)

display(selected_events_per_year)

,Year,Event
0,2020,70th Anniversary Grand Prix
673,2020,Abu Dhabi Grand Prix
1506,2020,Austrian Grand Prix
2145,2020,Bahrain Grand Prix
3642,2020,Belgian Grand Prix
14069,2021,Abu Dhabi Grand Prix
14628,2021,Austrian Grand Prix
15442,2021,Azerbaijan Grand Prix
16060,2021,Bahrain Grand Prix
16691,2021,Belgian Grand Prix


In [29]:
feature_cols = X_sim.columns.tolist()

base_rows = X_sim.merge(
    selected_events_per_year,
    on=["Year", "Event"],
    how="inner"
)

base_rows = base_rows[
    (base_rows["LapNumber"] % 10 == 0) &
    (base_rows["Rainfall"] == False) &
    (base_rows["Compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()

base_rows = base_rows.sort_values(["Year", "Event", "Driver", "LapNumber"])

display(base_rows[[
    "Year", "Event", "Driver", "Team", "LapNumber", "Stint",
    "Compound", "TyreLife", "TrackTemp", "Rainfall"
]])

,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall
51,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,10,2,HARD,4,43.227586,False
21,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,20,2,HARD,14,43.227586,False
36,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,30,2,HARD,24,43.227586,False
12,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,40,3,HARD,10,43.227586,False
2,2020,70th Anniversary Grand Prix,ALB,Red Bull Racing,50,3,HARD,20,43.227586,False
...,...,...,...,...,...,...,...,...,...,...
19646,2024,Bahrain Grand Prix,ZHO,Kick Sauber,10,2,HARD,1,23.652866,False
19633,2024,Bahrain Grand Prix,ZHO,Kick Sauber,20,2,HARD,11,23.652866,False
19623,2024,Bahrain Grand Prix,ZHO,Kick Sauber,30,3,HARD,2,23.652866,False
19613,2024,Bahrain Grand Prix,ZHO,Kick Sauber,40,3,HARD,12,23.652866,False


In [30]:
scenarios_driver_team = []

driver_team_pairs = (
    X_sim[["Driver", "Team"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["Team", "Driver"])
)

for _, base_row in base_rows.iterrows():
    for _, pair in driver_team_pairs.iterrows():
        row = base_row[feature_cols].copy()
        row["Driver"] = pair["Driver"]
        row["Team"] = pair["Team"]

        row_df = pd.DataFrame([row], columns=feature_cols)
        pred = model_sim.predict(row_df)[0]

        out = base_row.copy()
        out["Driver"] = pair["Driver"]
        out["Team"] = pair["Team"]
        out["scenario_type"] = "driver_team"
        out["scenario_value"] = f'{pair["Driver"]}_{pair["Team"]}'
        out["predicted_laptime"] = pred

        scenarios_driver_team.append(out)

sim_driver_team = pd.DataFrame(scenarios_driver_team)

display(
    sim_driver_team[[
        "Year", "Event", "Driver", "Team", "LapNumber", "Stint",
        "Compound", "TyreLife", "TrackTemp", "Rainfall",
        "predicted_laptime"
    ]].sort_values(["Year", "Event", "LapNumber", "predicted_laptime"])
)

,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,TrackTemp,Rainfall,predicted_laptime
1831,2020,70th Anniversary Grand Prix,HAM,Mercedes,10,2,HARD,2,43.227586,False,92.201882
1831,2020,70th Anniversary Grand Prix,BOT,Mercedes,10,2,HARD,2,43.227586,False,92.208449
1831,2020,70th Anniversary Grand Prix,RUS,Mercedes,10,2,HARD,2,43.227586,False,92.208449
515,2020,70th Anniversary Grand Prix,HAM,Mercedes,10,2,HARD,3,43.227586,False,92.336647
95,2020,70th Anniversary Grand Prix,HAM,Mercedes,10,2,HARD,3,43.227586,False,92.336647
...,...,...,...,...,...,...,...,...,...,...,...
19603,2024,Bahrain Grand Prix,MAZ,Haas F1 Team,50,3,HARD,22,23.652866,False,96.958506
19527,2024,Bahrain Grand Prix,MAZ,Haas F1 Team,50,3,HARD,23,23.652866,False,96.958506
19603,2024,Bahrain Grand Prix,MSC,Haas F1 Team,50,3,HARD,22,23.652866,False,96.958506
19527,2024,Bahrain Grand Prix,HUL,Haas F1 Team,50,3,HARD,23,23.652866,False,97.182201


In [32]:
import uuid

scenario_run_id = f"combo_laptime_{uuid.uuid4().hex[:10]}"
print("scenario_run_id:", scenario_run_id)

sim_combo_to_save = sim_driver_team.copy()

sim_combo_to_save = sim_combo_to_save.sort_values(
    ["Year", "Event", "LapNumber", "predicted_laptime"]
).reset_index(drop=True)

sim_combo_to_save["ranking"] = (
    sim_combo_to_save
    .groupby(["Year", "Event", "LapNumber"])["predicted_laptime"]
    .rank(method="first", ascending=True)
    .astype(int)
)

sim_combo_to_save["scenario_run_id"] = scenario_run_id
sim_combo_to_save["notes"] = "Comparación piloto-equipo con todos los datos"

# SOLO columnas que seguramente ya existen en simulation_scenarios
cols_to_save = [
    "scenario_run_id",
    "Year", "Event", "Driver", "Team", "LapNumber", "Stint",
    "Compound", "TyreLife", "FreshTyre", "TrackStatus",
    "AirTemp", "Humidity", "Pressure", "Rainfall",
    "TrackTemp", "WindDirection", "WindSpeed",
    "scenario_type", "scenario_value", "predicted_laptime"
]

for c in cols_to_save:
    if c not in sim_combo_to_save.columns:
        sim_combo_to_save[c] = None

sim_combo_to_save = sim_combo_to_save[cols_to_save]

print("Filas a guardar:", len(sim_combo_to_save))
display(sim_combo_to_save.head())

sim_combo_to_save.to_sql(
    "simulation_scenarios",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Todos los datos del escenario piloto-equipo fueron guardados en Supabase.")

scenario_run_id: combo_laptime_5c1fcd2a62
Filas a guardar: 128250


,scenario_run_id,Year,Event,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,...,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,scenario_type,scenario_value,predicted_laptime
0,combo_laptime_5c1fcd2a62,2020,70th Anniversary Grand Prix,HAM,Mercedes,10,2,HARD,2,True,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,driver_team,HAM_Mercedes,92.201882
1,combo_laptime_5c1fcd2a62,2020,70th Anniversary Grand Prix,BOT,Mercedes,10,2,HARD,2,True,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,driver_team,BOT_Mercedes,92.208449
2,combo_laptime_5c1fcd2a62,2020,70th Anniversary Grand Prix,RUS,Mercedes,10,2,HARD,2,True,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,driver_team,RUS_Mercedes,92.208449
3,combo_laptime_5c1fcd2a62,2020,70th Anniversary Grand Prix,HAM,Mercedes,10,2,HARD,3,True,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,driver_team,HAM_Mercedes,92.336647
4,combo_laptime_5c1fcd2a62,2020,70th Anniversary Grand Prix,HAM,Mercedes,10,2,HARD,3,True,...,25.287069,58.556034,1000.950862,False,43.227586,71.931034,2.236207,driver_team,HAM_Mercedes,92.336647


PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)